# 04 · A Negative-Result Diagnosis

A baseline 是负结果。这个 notebook 不解救 A，只回答：

1. 负结果是 broad-based，还是集中在少数 sector / era？
2. 换 K=5 → K=10（都是 ex-ante）后，结论是否改变？
3. 成本 0/5/10/20 bps 下，负结果是否只是成本造成的？

结论写成 **no support + wrong sign**，不写成 “proven absent”。

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


In [2]:
import pandas as pd
from IPython.display import display

from src import a_diagnostics as ad


## 1. Load the A baseline result

`output/carry_full_*` 是 `02_carry.ipynb` 已经跑出来的 full-sample result。


In [3]:
full = ad.load_result("output/carry_full")
print(full.positions.shape, full.daily_returns.shape, full.portfolio_returns.shape)


(468750, 11) (245806, 12) (10507, 6)


## 2. Side × sector × era

`raw_return = return_gross × direction` 恢复原始 spread return。
`contribution = return_gross / n_active`；所有格子加总 = 每日 portfolio gross return 的累计和。
成本不强行分摊到格子。


In [4]:
tab = ad.side_sector_era(full)
display(tab.loc[tab["level"] == "side"])
display(tab.loc[tab["level"] == "cell"].head(12))

pivot = tab.loc[tab["level"] == "cell"].pivot_table(
    index=["side", "sector"], columns="era", values="raw_mean"
)
display(pivot.round(6))


,side,sector,era,n_obs,n_dates,raw_mean,t_stat,contribution,avg_n_active,level
42,long,NaN,NaN,73862,10373,-0.000008,-0.650509,-0.032889,22.713926,side
43,short,NaN,NaN,160513,10371,0.000024,2.463452,-0.184489,22.969872,side


,side,sector,era,n_obs,n_dates,raw_mean,t_stat,contribution,avg_n_active,level
0,long,Energy,1986-1999,5633,2719,0.000149,1.055128,0.044136,21.707083,cell
1,long,Energy,2000-2009,3705,1790,0.000031,0.968111,0.009518,22.880162,cell
2,long,Energy,2010-2026,10336,3884,0.000050,1.342910,0.002041,23.355844,cell
3,long,Grains,1986-1999,2995,2020,-0.000044,-0.901936,-0.005352,21.315860,cell
4,long,Grains,2000-2009,1205,982,-0.000270,-1.697948,-0.013803,22.933610,cell
5,long,Grains,2010-2026,3248,2234,-0.000121,-1.019207,-0.016776,23.657943,cell
6,long,Industrials,1986-1999,2794,2179,-0.000070,-1.021736,-0.007333,21.432713,cell
7,long,Industrials,2000-2009,853,814,-0.000294,-2.052644,-0.011207,22.876905,cell
8,long,Industrials,2010-2026,3101,2471,-0.000145,-1.798840,-0.018955,23.763302,cell
9,long,Livestock,1986-1999,6229,3263,-0.000030,-0.866291,-0.007709,21.508589,cell


era                1986-1999  2000-2009  2010-2026
side  sector                                      
long  Energy        0.000149   0.000031   0.000050
      Grains       -0.000044  -0.000270  -0.000121
      Industrials  -0.000070  -0.000294  -0.000145
      Livestock    -0.000030  -0.000117   0.000015
      Metals       -0.000068  -0.000121  -0.000027
      Oilseeds      0.000070  -0.000002   0.000085
      Softs         0.000032  -0.000041  -0.000033
short Energy        0.000083   0.000059   0.000051
      Grains       -0.000021  -0.000040   0.000020
      Industrials   0.000018   0.000016   0.000130
      Livestock     0.000060  -0.000029   0.000033
      Metals       -0.000001  -0.000012  -0.000003
      Oilseeds      0.000011   0.000005   0.000004
      Softs         0.000037   0.000045   0.000085

## 3. K=5 vs K=10（ex-ante）

K=10 是 `days_to_delivery_start >= 10`，和 K=5 一样只用事前信息。
D=10 不跑：它依赖 `last_trade_proxy`，有 look-ahead。


In [5]:
rank = pd.read_parquet("data/processed/rank.parquet")
elig = ad.eligibility_compare(rank, k_values=(5, 10), results={5: full})
elig.to_csv("output/a_eligibility_compare.csv", index=False)
display(elig)


,k,pair_rows,m1_m2,m2_m3,ann_return_gross,ann_return_net,sharpe_gross,sharpe_net,max_drawdown_net,long_raw_mean,short_raw_mean,long_t_stat,short_t_stat
0,5,236385,159817,76568,-0.005309,-0.018968,-0.022104,-0.079878,-0.553106,-0.000008,0.000024,-0.650509,2.463452
1,10,234934,132544,102390,-0.006072,-0.019424,-0.026635,-0.085329,-0.559863,-0.000032,0.000018,-2.825488,1.840338


## 4. Cost sensitivity

Baseline 的成本序列是 5 bps 的；net cost 对 bps 线性，所以直接缩放，不重跑。


In [6]:
cost = ad.cost_sensitivity(full)
cost.to_csv("output/a_cost_sensitivity.csv", index=False)
display(cost)


,cost_bps_per_leg,ann_return_net,ann_vol_net,sharpe_net,max_drawdown_net
0,0.0,-0.005309,0.014858,-0.022104,-0.226338
1,5.0,-0.018968,0.015014,-0.079878,-0.553106
2,10.0,-0.032445,0.015478,-0.133743,-0.747214
3,20.0,-0.058861,0.017217,-0.221390,-0.920292


**Zero-cost check:** `0 bps` 这一行就是**不考虑 transaction cost** 的结果。
Full-sample gross ann. −0.53%，gross Sharpe −0.02；A 在成本前已经是负的。
5 bps 的 net 结果是 −1.90% / −0.08，成本只是放大亏损，不是负号的原因。

## 5. 读结果

**Side：**

- Long（backwardation）：raw mean 约 0，t 不显著；long spread 没有补偿。
- Short（contango）：raw mean 为正，t 约 +2.5；short spread 是错方向。
- 负 P&L 主要来自 short side。

**K=5 vs K=10：**

- 两者 gross/net Sharpe 都为负；K=10 没有翻正。
- K=10 的 short raw mean 仍为正（t 约 +1.8），long raw mean 更负（t 约 −2.8）。
- 所以负结果不是 K=5 buffer 的偶然。

**成本：**

- 0 bps 时 gross ann. 已经约 −0.53%，Sharpe 约 −0.02。
- 5 bps → −1.90%；20 bps → −5.89%。
- 成本放大负结果，但不是负号的原因。

**结论：**

> A 在这个 spec 下没有支持：两个 side 的 raw mean 都不支持假设，
> contango short 显著错方向；K=10 和成本变化不翻符号。
> Test 期 t 只有 +0.87，所以严格说是 **no evidence + wrong sign in-sample**，
> 不是数学意义上的 “proven absent”。

机制上，contango 里 near 相对便宜；随着 near 接近到期，它向 spot 收敛，
spread（near − far）倾向于上升，所以 short spread 方向错。这不是
“再调一个参数就能救”的问题。


## 6. 下一步

- 不跑 rescue 变体（long-only / threshold / cross-section）。
- A 以 negative baseline 收尾，记录在 `docs/a_negative_result.md`。
- C 继续在另一个 branch 上推进；本 branch 只保留诊断文件。